# HYPERVIEW2 Compression Diagnostics

Notebook analizuje zapisane wyniki downstream i rekonstrukcje HYPERVIEW2 z Google Drive. Nie generuje rekonstrukcji od nowa i nie instaluje Mamby. Celem jest odpowiedziec na pytania:

1. jak duzy jest domain shift `original -> reconstruction`,
2. czy retrening downstream na rekonstrukcjach odzyskuje wynik,
3. ktore targety i pasma najbardziej cierpia,
4. czy normalizacja cech (`none` vs `percentile`) zmienia wnioski.


## 1. Repo i lekkie zaleznosci

Ta komorka klonuje/aktualizuje repo, ale nie wykonuje `pip install -e .`, zeby nie ruszac zaleznosci CompressAI i Mamby w Colabie. Kod downstream jest importowany bezposrednio z `src`.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/mhx1467/master-thesis-code.git'
REPO_DIR = Path('/content/hsi')
REPO_REF = 'main'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR, check=True)

for module_name in list(sys.modules):
    if module_name == 'hsi_compression' or module_name.startswith('hsi_compression.'):
        del sys.modules[module_name]

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo:', Path.cwd())
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())



In [ ]:
%pip -q install pandas numpy matplotlib scikit-learn tqdm tifffile



## 2. Sciezki na Google Drive

Zakladany layout jest zgodny z notebookiem uruchomieniowym:

- `MyDrive/hsi/data/hyperview2/HYPERVIEW2`
- `MyDrive/hsi/reconstructions/hyperview2/<variant>/HYPERVIEW2`
- `MyDrive/hsi/downstream_results/hyperview2_compression`


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_HSI = Path('/content/drive/MyDrive/hsi')
HV2_ROOT = DRIVE_HSI / 'data/hyperview2/HYPERVIEW2'
RECON_PARENT = DRIVE_HSI / 'reconstructions/hyperview2'
COMPRESSION_RESULTS_DIR = DRIVE_HSI / 'downstream_results/hyperview2_compression'
DIAG_DIR = DRIVE_HSI / 'downstream_results/hyperview2_diagnostics'

SUMMARY_CSV = COMPRESSION_RESULTS_DIR / 'compression_downstream_summary.csv'
METRICS_JSON = COMPRESSION_RESULTS_DIR / 'compression_downstream_metrics.json'
PREDICTIONS_CSV = COMPRESSION_RESULTS_DIR / 'compression_downstream_predictions.csv'

DIAG_DIR.mkdir(parents=True, exist_ok=True)

print('HV2_ROOT:', HV2_ROOT, HV2_ROOT.exists())
print('RECON_PARENT:', RECON_PARENT, RECON_PARENT.exists())
print('SUMMARY_CSV:', SUMMARY_CSV, SUMMARY_CSV.exists())
print('METRICS_JSON:', METRICS_JSON, METRICS_JSON.exists())
print('PREDICTIONS_CSV:', PREDICTIONS_CSV, PREDICTIONS_CSV.exists())
print('DIAG_DIR:', DIAG_DIR)



## 3. Importy i konfiguracja


In [ ]:
import json
import math
import time
from dataclasses import dataclass
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from hsi_compression.downstream import (
    HYPERVIEW2_TARGET_COLUMNS,
    Hyperview2FeatureDataset,
    build_hyperview2_regressor,
    build_hyperview2_samples,
    compute_regression_metrics,
    split_samples,
)
from hsi_compression.downstream.hyperview2 import load_array, load_mask, normalize_cube, to_chw

plt.rcParams['figure.dpi'] = 130
TARGETS = list(HYPERVIEW2_TARGET_COLUMNS)
MODALITY = 'prisma'
FEATURE_SET = 'mean_std_derivatives'
SEED = 42
VAL_FRACTION = 0.2
N_JOBS = -1

MODEL_NAMES = [
    'dummy_mean',
    'ridge',
    'extra_trees',
    'random_forest',
    'hist_gradient_boosting',
]

NORMALIZATION_GRID = ['none', 'percentile']
RUN_NORMALIZATION_SWEEP = True
RUN_FEATURE_DRIFT = True
MAX_SPECTRAL_SAMPLES = None  # set np. 300 for faster diagnostics
MAX_FEATURE_DRIFT_SAMPLES = None  # set np. 300 for faster diagnostics




## 4. Podsumowanie zapisanych wynikow downstream


In [ ]:
if not SUMMARY_CSV.exists():
    raise FileNotFoundError(f'Missing summary CSV: {SUMMARY_CSV}')

summary_df = pd.read_csv(SUMMARY_CSV)
# The compression notebook currently writes `variant`; older drafts of this diagnostics
# notebook used `source`. Keep both names available so archived CSVs remain readable.
if 'source' not in summary_df.columns:
    if 'variant' in summary_df.columns:
        summary_df = summary_df.rename(columns={'variant': 'source'})
    elif 'model_variant' in summary_df.columns:
        summary_df = summary_df.rename(columns={'model_variant': 'source'})
    else:
        raise KeyError(f'Cannot infer source/variant column. Columns: {list(summary_df.columns)}')
if 'variant' not in summary_df.columns:
    summary_df['variant'] = summary_df['source']

print('Rows:', len(summary_df))
print('Columns:', list(summary_df.columns))
display(summary_df.sort_values(['source', 'mode', 'hyperview_score']).reset_index(drop=True))

predictions_df = pd.DataFrame()
if PREDICTIONS_CSV.exists():
    predictions_df = pd.read_csv(PREDICTIONS_CSV)
    if 'source' not in predictions_df.columns and 'variant' in predictions_df.columns:
        predictions_df = predictions_df.rename(columns={'variant': 'source'})
    if 'variant' not in predictions_df.columns and 'source' in predictions_df.columns:
        predictions_df['variant'] = predictions_df['source']
    print('Prediction rows:', len(predictions_df))
    display(predictions_df.head())
else:
    print('Prediction-level CSV missing. Rerun the main compression downstream notebook to generate it:', PREDICTIONS_CSV)


In [ ]:
def best_by_mode(df: pd.DataFrame) -> pd.DataFrame:
    ok = df[df['status'].eq('ok') & df['hyperview_score'].notna()].copy()
    idx = ok.groupby(['source', 'mode'])['hyperview_score'].idxmin()
    return ok.loc[idx].sort_values(['source', 'mode']).reset_index(drop=True)

best_df = best_by_mode(summary_df)
display(best_df[['source', 'mode', 'model', 'hyperview_score', 'mean_mse', 'mean_mae', 'fit_time_sec', 'predict_time_sec']])

original_best = best_df.loc[best_df['source'].eq('original'), 'hyperview_score'].min()
rows = []
for _, row in best_df[~best_df['source'].eq('original')].iterrows():
    rows.append({
        'source': row['source'],
        'mode': row['mode'],
        'best_model': row['model'],
        'hyperview_score': row['hyperview_score'],
        'ratio_to_best_original': row['hyperview_score'] / original_best if original_best else np.nan,
    })
shift_df = pd.DataFrame(rows)
display(shift_df)



In [ ]:
plot_df = best_df[['source', 'mode', 'model', 'hyperview_score']].copy()
plot_df['label'] = plot_df['source'] + '\n' + plot_df['mode'] + '\n' + plot_df['model']
fig, ax = plt.subplots(figsize=(max(8, 0.55 * len(plot_df)), 4))
colors = ['#4c78a8' if src == 'original' else '#f58518' for src in plot_df['source']]
ax.bar(np.arange(len(plot_df)), plot_df['hyperview_score'], color=colors)
ax.axhline(1.0, color='black', linewidth=1, linestyle='--', label='dummy mean')
ax.set_ylabel('Hyperview score (lower is better)')
ax.set_xticks(np.arange(len(plot_df)))
ax.set_xticklabels(plot_df['label'], rotation=60, ha='right')
ax.set_title('Best downstream score by source and mode')
ax.legend(loc='upper right')
fig.tight_layout()
out = DIAG_DIR / 'best_downstream_scores.png'
fig.savefig(out, bbox_inches='tight')
print('Saved:', out)
plt.show()



## 5. Podsumowanie jakosci rekonstrukcji

Ta sekcja zbiera `reconstruction_summary.json` z katalogow rekonstrukcji. Jesli masz kilka wariantow Mamby, pojawia sie tu obok siebie.


In [ ]:
def discover_recon_roots(parent: Path) -> dict[str, Path]:
    roots = {}
    if parent.exists():
        for path in sorted(parent.glob('*/HYPERVIEW2')):
            if (path / 'train/hsi_satellite').is_dir():
                roots[path.parent.name] = path
    return roots


def infer_recon_input_normalization(name: str, payload: dict[str, Any] | None) -> str:
    if payload:
        value = payload.get('input_normalization')
        if value:
            return str(value)
    if name.endswith('_input_percentile'):
        return 'percentile'
    if name.endswith('_input_minmax'):
        return 'minmax'
    return 'none'


RECON_ROOTS = discover_recon_roots(RECON_PARENT)
RECON_INPUT_NORMALIZATIONS: dict[str, str] = {}
print('Recon roots:')
for name, root in RECON_ROOTS.items():
    print(' ', name, '->', root)

summary_rows = []
for name, root in RECON_ROOTS.items():
    summary_path = root / 'reconstruction_summary.json'
    payload = json.loads(summary_path.read_text(encoding='utf-8')) if summary_path.exists() else None
    input_normalization = infer_recon_input_normalization(name, payload)
    RECON_INPUT_NORMALIZATIONS[name] = input_normalization
    row = {
        'source': name,
        'recon_root': str(root),
        'summary_exists': summary_path.exists(),
        'input_normalization': input_normalization,
    }
    if payload:
        for key in [
            'variant',
            'saved_reconstruction_normalization',
            'reconstruction_value_space',
            'samples',
            'metric_samples',
            'masked_psnr',
            'masked_sam_deg',
            'masked_mse',
            'masked_mae',
            'actual_bpppc',
            'actual_cr_16bit',
            'encode_time_sec',
            'decode_time_sec',
        ]:
            row[key] = payload.get(key)
    summary_rows.append(row)

recon_summary_df = pd.DataFrame(summary_rows)
display(recon_summary_df)


## 6. Diagnostyka spektralna oryginal vs rekonstrukcja

Liczymy blad per band na walidacyjnej czesci `train_gt.csv` zgodnej z poprzednim notebookiem. Dla masek per-band uzywamy maski wartosci, a nie tylko maski przestrzennej.


In [ ]:
def safe_sample_stem(sample_id: str) -> str:
    return f'{int(sample_id):04d}' if str(sample_id).isdigit() else str(sample_id)


def load_cube_and_value_mask(path: Path, modality: str = 'prisma') -> tuple[np.ndarray, np.ndarray]:
    cube = load_array(path, modality=modality)
    with np.load(path) as archive:
        mask_arr = np.asarray(archive['mask']) if 'mask' in archive.files else None
    if mask_arr is None:
        value_mask = np.isfinite(cube)
    else:
        mask_arr = np.asarray(mask_arr)
        if mask_arr.shape == cube.shape:
            value_mask = mask_arr.astype(bool)
        else:
            spatial = load_mask(path, shape_hw=tuple(cube.shape[-2:]))
            value_mask = np.broadcast_to(spatial[None], cube.shape).copy()
    return cube.astype(np.float32, copy=False), value_mask.astype(bool, copy=False)


def normalize_original_cube(cube: np.ndarray, value_mask: np.ndarray, normalization: str) -> np.ndarray:
    if normalization == 'none':
        return np.nan_to_num(cube, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    spatial_mask = value_mask.any(axis=0)
    return normalize_cube(cube, mask=spatial_mask, mode=normalization)


def sample_path(root: Path, sample_id: str, split: str = 'train') -> Path:
    return root / split / 'hsi_satellite' / f'{safe_sample_stem(sample_id)}.npz'


def mean_spectrum(cube: np.ndarray, mask: np.ndarray) -> np.ndarray:
    counts = mask.sum(axis=(1, 2)).astype(np.float32)
    values = (cube * mask).sum(axis=(1, 2)).astype(np.float32)
    out = np.full(cube.shape[0], np.nan, dtype=np.float32)
    np.divide(values, counts, out=out, where=counts > 0)
    return out


def compute_spectral_diagnostics(
    original_root: Path,
    recon_root: Path,
    sample_ids: list[str],
    original_normalization: str = 'none',
    max_samples: int | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if max_samples is not None:
        sample_ids = sample_ids[:max_samples]
    n_bands = 230
    abs_sum = np.zeros(n_bands, dtype=np.float64)
    sq_sum = np.zeros(n_bands, dtype=np.float64)
    bias_sum = np.zeros(n_bands, dtype=np.float64)
    orig_sum = np.zeros(n_bands, dtype=np.float64)
    recon_sum = np.zeros(n_bands, dtype=np.float64)
    counts = np.zeros(n_bands, dtype=np.float64)
    sample_rows = []

    for sample_id in tqdm(sample_ids, desc=f'spectral:{recon_root.parent.name}:{original_normalization}'):
        orig_path = sample_path(original_root, sample_id)
        recon_path = sample_path(recon_root, sample_id)
        if not orig_path.exists() or not recon_path.exists():
            continue
        orig, orig_mask = load_cube_and_value_mask(orig_path)
        recon, recon_mask = load_cube_and_value_mask(recon_path)
        orig = normalize_original_cube(orig, orig_mask, original_normalization)
        c = min(orig.shape[0], recon.shape[0])
        h = min(orig.shape[-2], recon.shape[-2])
        w = min(orig.shape[-1], recon.shape[-1])
        orig = orig[:c, :h, :w]
        recon = recon[:c, :h, :w]
        valid = orig_mask[:c, :h, :w] & recon_mask[:c, :h, :w]
        if not valid.any():
            sample_rows.append({'sample_id': sample_id, 'valid_values': 0, 'mae': np.nan, 'rmse': np.nan})
            continue
        diff = recon - orig
        valid_f = valid.astype(np.float32)
        band_counts = valid_f.sum(axis=(1, 2))
        abs_sum[:c] += (np.abs(diff) * valid_f).sum(axis=(1, 2))
        sq_sum[:c] += ((diff ** 2) * valid_f).sum(axis=(1, 2))
        bias_sum[:c] += (diff * valid_f).sum(axis=(1, 2))
        orig_sum[:c] += (orig * valid_f).sum(axis=(1, 2))
        recon_sum[:c] += (recon * valid_f).sum(axis=(1, 2))
        counts[:c] += band_counts
        sample_rows.append({
            'sample_id': sample_id,
            'valid_values': int(valid.sum()),
            'original_normalization': original_normalization,
            'mae': float(np.abs(diff[valid]).mean()),
            'rmse': float(np.sqrt((diff[valid] ** 2).mean())),
        })

    band = np.arange(n_bands)
    per_band = pd.DataFrame({
        'band': band,
        'valid_count': counts,
        'original_normalization': original_normalization,
        'orig_mean': np.divide(orig_sum, counts, out=np.full(n_bands, np.nan), where=counts > 0),
        'recon_mean': np.divide(recon_sum, counts, out=np.full(n_bands, np.nan), where=counts > 0),
        'bias': np.divide(bias_sum, counts, out=np.full(n_bands, np.nan), where=counts > 0),
        'mae': np.divide(abs_sum, counts, out=np.full(n_bands, np.nan), where=counts > 0),
        'rmse': np.sqrt(np.divide(sq_sum, counts, out=np.full(n_bands, np.nan), where=counts > 0)),
    })
    return per_band, pd.DataFrame(sample_rows)


In [ ]:
with METRICS_JSON.open('r', encoding='utf-8') as handle:
    metrics_payload = json.load(handle)

val_ids = [str(item) for item in metrics_payload.get('protocol', {}).get('val_sample_ids', [])]
if not val_ids:
    all_samples = build_hyperview2_samples(HV2_ROOT, modality=MODALITY, split='train')
    _, val_samples = split_samples(all_samples, val_fraction=VAL_FRACTION, seed=SEED)
    val_ids = [sample.sample_id for sample in val_samples]

print('Validation sample ids:', len(val_ids))
print('First ids:', val_ids[:8])



In [ ]:
per_band_tables = {}
sample_error_tables = {}

for name, recon_root in RECON_ROOTS.items():
    original_normalization = RECON_INPUT_NORMALIZATIONS.get(name, 'none')
    per_band, sample_errors = compute_spectral_diagnostics(
        HV2_ROOT,
        recon_root,
        val_ids,
        original_normalization=original_normalization,
        max_samples=MAX_SPECTRAL_SAMPLES,
    )
    per_band['source'] = name
    sample_errors['source'] = name
    per_band_tables[name] = per_band
    sample_error_tables[name] = sample_errors
    per_band_path = DIAG_DIR / f'{name}_per_band_diagnostics.csv'
    sample_path_out = DIAG_DIR / f'{name}_sample_errors.csv'
    per_band.to_csv(per_band_path, index=False)
    sample_errors.to_csv(sample_path_out, index=False)
    print('Saved:', per_band_path)
    print('Saved:', sample_path_out)

if per_band_tables:
    all_per_band_df = pd.concat(per_band_tables.values(), ignore_index=True)
    all_per_band_path = DIAG_DIR / 'per_band_diagnostics_all.csv'
    all_per_band_df.to_csv(all_per_band_path, index=False)
    print('Saved:', all_per_band_path)
    display(all_per_band_df.groupby(['source', 'original_normalization'])[['mae', 'rmse', 'bias']].mean().reset_index())
else:
    print('No reconstruction roots found.')


In [ ]:
if per_band_tables:
    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    for name, per_band in per_band_tables.items():
        axes[0].plot(per_band['band'], per_band['mae'], label=name)
        axes[1].plot(per_band['band'], per_band['bias'], label=name)
    axes[0].set_ylabel('MAE')
    axes[0].set_title('Per-band reconstruction error')
    axes[1].set_ylabel('Bias: recon - original')
    axes[1].set_xlabel('Band index')
    for ax in axes:
        ax.grid(True, alpha=0.25)
        ax.legend()
    fig.tight_layout()
    out = DIAG_DIR / 'per_band_error_curves.png'
    fig.savefig(out, bbox_inches='tight')
    print('Saved:', out)
    plt.show()



In [ ]:
def plot_sample_spectra(
    original_root: Path,
    recon_root: Path,
    sample_ids: list[str],
    title: str,
    out_path: Path,
    original_normalization: str = 'none',
):
    n = min(len(sample_ids), 6)
    if n == 0:
        print('No sample ids to plot')
        return
    fig, axes = plt.subplots(n, 2, figsize=(12, 2.5 * n), sharex=True)
    if n == 1:
        axes = np.asarray([axes])
    for row, sample_id in enumerate(sample_ids[:n]):
        orig, orig_mask = load_cube_and_value_mask(sample_path(original_root, sample_id))
        recon, recon_mask = load_cube_and_value_mask(sample_path(recon_root, sample_id))
        orig = normalize_original_cube(orig, orig_mask, original_normalization)
        c = min(orig.shape[0], recon.shape[0])
        h = min(orig.shape[-2], recon.shape[-2])
        w = min(orig.shape[-1], recon.shape[-1])
        valid = orig_mask[:c, :h, :w] & recon_mask[:c, :h, :w]
        orig_spec = mean_spectrum(orig[:c, :h, :w], valid)
        recon_spec = mean_spectrum(recon[:c, :h, :w], valid)
        band = np.arange(c)
        axes[row, 0].plot(band, orig_spec, label=f'original ({original_normalization})', linewidth=1.4)
        axes[row, 0].plot(band, recon_spec, label='reconstruction', linewidth=1.1)
        axes[row, 0].set_ylabel(sample_id)
        axes[row, 0].grid(True, alpha=0.25)
        axes[row, 1].plot(band, np.abs(recon_spec - orig_spec), color='#e45756', linewidth=1.1)
        axes[row, 1].grid(True, alpha=0.25)
    axes[0, 0].set_title('Mean spectrum')
    axes[0, 1].set_title('Absolute spectral error')
    axes[-1, 0].set_xlabel('Band index')
    axes[-1, 1].set_xlabel('Band index')
    axes[0, 0].legend(loc='best')
    fig.suptitle(title, y=1.0)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches='tight')
    print('Saved:', out_path)
    plt.show()

for name, errors in sample_error_tables.items():
    root = RECON_ROOTS[name]
    original_normalization = RECON_INPUT_NORMALIZATIONS.get(name, 'none')
    worst_ids = errors.sort_values('mae', ascending=False)['sample_id'].dropna().astype(str).tolist()[:6]
    out = DIAG_DIR / f'{name}_worst_sample_spectra.png'
    plot_sample_spectra(
        HV2_ROOT,
        root,
        worst_ids,
        f'Worst validation spectra: {name}',
        out,
        original_normalization=original_normalization,
    )


## 7. Per-target degradacja downstream

Tabela ponizej korzysta z zapisanych wynikow CSV. Warto patrzec osobno na `original_train_to_recon_val` i `recon_train_to_recon_val`, bo odpowiadaja na inne pytania.


In [ ]:
rel_cols = [col for col in summary_df.columns if col.endswith('_relative_mse')]
rmse_cols = [col for col in summary_df.columns if col.endswith('_rmse')]
focus = summary_df[summary_df['status'].eq('ok')].copy()
cols = ['source', 'mode', 'model', 'hyperview_score', *rel_cols]
display(focus[cols].sort_values(['source', 'mode', 'hyperview_score']).reset_index(drop=True))

best_rows = best_by_mode(summary_df)
fig, axes = plt.subplots(len(best_rows), 1, figsize=(9, max(3, 2.2 * len(best_rows))), sharex=True)
if len(best_rows) == 1:
    axes = [axes]
for ax, (_, row) in zip(axes, best_rows.iterrows()):
    vals = [row.get(f'{target}_relative_mse', np.nan) for target in TARGETS]
    ax.bar(TARGETS, vals, color='#72b7b2')
    ax.axhline(1.0, color='black', linestyle='--', linewidth=1)
    ax.set_ylabel('rel. MSE')
    ax.set_title(f"{row['source']} | {row['mode']} | {row['model']} | score={row['hyperview_score']:.3f}")
    ax.grid(True, axis='y', alpha=0.25)
axes[-1].set_xlabel('Target')
fig.tight_layout()
out = DIAG_DIR / 'per_target_relative_mse_best_models.png'
fig.savefig(out, bbox_inches='tight')
print('Saved:', out)
plt.show()



## 8. Diagnostyka predykcji i domain shift

Ta sekcja korzysta z pelnej tabeli predykcji zapisanej przez glowny notebook. Rozklada blad rekonstrukcji na przesuniecie predykcji wzgledem predykcji na oryginale oraz pokazuje, ktore targety i modele sa najbardziej wrazliwe.


In [ ]:
if predictions_df.empty:
    print('No prediction-level data available. Rerun the main notebook after pulling the latest repo.')
else:
    prediction_metrics = (
        predictions_df
        .groupby(['source', 'mode', 'model', 'target'], dropna=False)
        .agg(
            samples=('sample_id', 'nunique'),
            bias=('error', 'mean'),
            mae=('abs_error', 'mean'),
            mse=('squared_error', 'mean'),
            relative_mse=('relative_squared_error', 'mean'),
        )
        .reset_index()
    )
    prediction_metrics['rmse'] = np.sqrt(prediction_metrics['mse'])
    out = DIAG_DIR / 'prediction_metrics_by_target.csv'
    prediction_metrics.to_csv(out, index=False)
    print('Saved:', out)
    display(prediction_metrics.sort_values(['source', 'mode', 'model', 'relative_mse']).reset_index(drop=True))

    original_predictions = predictions_df[predictions_df['source'].eq('original') & predictions_df['mode'].eq('original_train_to_original_val')]
    recon_predictions = predictions_df[~predictions_df['source'].eq('original') & predictions_df['mode'].eq('original_train_to_recon_val')]
    joined = recon_predictions.merge(
        original_predictions[['model', 'sample_id', 'target', 'y_pred', 'error', 'abs_error', 'squared_error']],
        on=['model', 'sample_id', 'target'],
        how='inner',
        suffixes=('_recon', '_original'),
    )
    if joined.empty:
        print('No matching original/reconstruction predictions to compare.')
    else:
        joined['prediction_shift'] = joined['y_pred_recon'] - joined['y_pred_original']
        joined['abs_prediction_shift'] = joined['prediction_shift'].abs()
        joined['extra_abs_error'] = joined['abs_error_recon'] - joined['abs_error_original']
        joined['extra_squared_error'] = joined['squared_error_recon'] - joined['squared_error_original']
        joined['cross_term'] = 2.0 * joined['error_original'] * joined['prediction_shift']
        decomp = (
            joined
            .groupby(['source', 'model', 'target'], dropna=False)
            .agg(
                samples=('sample_id', 'nunique'),
                original_mse=('squared_error_original', 'mean'),
                recon_mse=('squared_error_recon', 'mean'),
                extra_mse=('extra_squared_error', 'mean'),
                shift_mse=('prediction_shift', lambda s: float(np.mean(np.asarray(s) ** 2))),
                cross_term=('cross_term', 'mean'),
                mean_shift=('prediction_shift', 'mean'),
                mean_abs_shift=('abs_prediction_shift', 'mean'),
                mean_extra_abs_error=('extra_abs_error', 'mean'),
            )
            .reset_index()
        )
        decomp['mse_identity_residual'] = decomp['recon_mse'] - (decomp['original_mse'] + decomp['shift_mse'] + decomp['cross_term'])
        out = DIAG_DIR / 'prediction_shift_decomposition.csv'
        decomp.to_csv(out, index=False)
        print('Saved:', out)
        display(decomp.sort_values(['source', 'model', 'extra_mse'], ascending=[True, True, False]).reset_index(drop=True))

        corr_rows = []
        for keys, group in joined.groupby(['source', 'model', 'target'], dropna=False):
            if len(group) < 2 or group['y_pred_original'].std() == 0 or group['y_pred_recon'].std() == 0:
                corr = np.nan
            else:
                corr = float(group['y_pred_original'].corr(group['y_pred_recon']))
            corr_rows.append({
                'source': keys[0],
                'model': keys[1],
                'target': keys[2],
                'prediction_corr_original_vs_recon': corr,
                'mean_abs_prediction_shift': float(group['abs_prediction_shift'].mean()),
                'mean_extra_abs_error': float(group['extra_abs_error'].mean()),
            })
        corr_df = pd.DataFrame(corr_rows)
        out = DIAG_DIR / 'prediction_shift_correlations.csv'
        corr_df.to_csv(out, index=False)
        print('Saved:', out)
        display(corr_df.sort_values(['source', 'model', 'mean_abs_prediction_shift'], ascending=[True, True, False]).reset_index(drop=True))

        degraded_best = best_df[(~best_df['source'].eq('original')) & best_df['mode'].eq('original_train_to_recon_val')]
        for _, best_row in degraded_best.iterrows():
            source = best_row['source']
            model = best_row['model']
            subset = joined[joined['source'].eq(source) & joined['model'].eq(model)].copy()
            if subset.empty:
                continue
            targets = list(TARGETS)
            fig, axes = plt.subplots(2, 3, figsize=(11, 7))
            axes = axes.ravel()
            for ax, target in zip(axes, targets):
                target_df = subset[subset['target'].eq(target)]
                ax.scatter(target_df['y_pred_original'], target_df['y_pred_recon'], s=10, alpha=0.55)
                low = float(np.nanmin([target_df['y_pred_original'].min(), target_df['y_pred_recon'].min()]))
                high = float(np.nanmax([target_df['y_pred_original'].max(), target_df['y_pred_recon'].max()]))
                ax.plot([low, high], [low, high], color='black', linewidth=0.8, linestyle='--')
                ax.set_title(target)
                ax.grid(True, alpha=0.2)
            fig.suptitle(f'Prediction shift: {source} | {model}')
            fig.supxlabel('prediction on original validation features')
            fig.supylabel('prediction on reconstructed validation features')
            fig.tight_layout()
            out = DIAG_DIR / f'{source}_{model}_prediction_shift_scatter.png'
            fig.savefig(out, bbox_inches='tight')
            print('Saved:', out)
            plt.show()


## 9. Dryft cech rekonstrukcji

Ta sekcja porownuje wektory cech downstream dla oryginalu i rekonstrukcji oraz sprawdza, czy probki z wiekszym dryftem cech maja wiekszy blad predykcji.


In [ ]:
def make_feature_matrix(samples, modality: str, normalization: str, feature_set: str):
    dataset = Hyperview2FeatureDataset(
        samples,
        modality=modality,
        normalization=normalization,
        feature_set=feature_set,
    )
    xs, ys, ids = [], [], []
    for idx in tqdm(range(len(dataset)), desc=f'features:{modality}:{normalization}:{feature_set}'):
        item = dataset[idx]
        xs.append(item['features'].numpy())
        ys.append(item['target'].numpy())
        ids.append(str(item['sample_id']))
    return np.stack(xs).astype(np.float32), np.stack(ys).astype(np.float32), ids


def samples_by_ids(root: Path, sample_ids: list[str], modality: str):
    samples = build_hyperview2_samples(root, modality=modality, split='train')
    by_id = {sample.sample_id: sample for sample in samples}
    missing = [sample_id for sample_id in sample_ids if sample_id not in by_id]
    if missing:
        raise KeyError(f'Missing samples in {root}: {missing[:8]}')
    return [by_id[sample_id] for sample_id in sample_ids]


def run_regressors(x_train, y_train, x_val, y_val, baseline_mse, model_names, source, mode, normalization, recon_feature_normalization=None):
    rows = []
    for name in model_names:
        start = time.perf_counter()
        row = {'source': source, 'mode': mode, 'normalization': normalization, 'recon_feature_normalization': recon_feature_normalization, 'model': name}
        try:
            model = build_hyperview2_regressor(
                name,
                random_state=SEED,
                n_jobs=N_JOBS,
                n_features=x_train.shape[1],
                n_samples=x_train.shape[0],
                n_targets=y_train.shape[1],
            )
            model.fit(x_train, y_train)
            fit_time = time.perf_counter() - start
            pred_start = time.perf_counter()
            y_pred = np.asarray(model.predict(x_val), dtype=np.float32)
            predict_time = time.perf_counter() - pred_start
            metrics = compute_regression_metrics(y_val, y_pred, baseline_mse)
            row.update({
                'status': 'ok',
                'hyperview_score': metrics['hyperview_score'],
                'mean_mse': metrics['mean_mse'],
                'mean_mae': metrics['mean_mae'],
                'fit_time_sec': fit_time,
                'predict_time_sec': predict_time,
            })
            for target, target_metrics in metrics['targets'].items():
                row[f'{target}_rmse'] = target_metrics['rmse']
                row[f'{target}_relative_mse'] = target_metrics['relative_mse']
        except Exception as exc:
            row.update({'status': 'failed', 'error': str(exc), 'fit_time_sec': time.perf_counter() - start})
        rows.append(row)
    return rows





In [ ]:
feature_drift_df = pd.DataFrame()
if not RUN_FEATURE_DRIFT:
    print('RUN_FEATURE_DRIFT=False')
elif not RECON_ROOTS:
    print('No reconstruction roots available for feature drift diagnostics.')
else:
    protocol = metrics_payload.get('protocol', {}) if 'metrics_payload' in globals() else {}
    original_norm = protocol.get('resolved_original_feature_normalization', 'percentile')
    recon_feature_norms = protocol.get('recon_feature_normalizations', {})
    original_samples = build_hyperview2_samples(HV2_ROOT, modality=MODALITY, split='train')
    train_samples, val_samples = split_samples(original_samples, val_fraction=VAL_FRACTION, seed=SEED)
    if MAX_FEATURE_DRIFT_SAMPLES is not None:
        val_samples = val_samples[:MAX_FEATURE_DRIFT_SAMPLES]
    val_ids_for_drift = [sample.sample_id for sample in val_samples]
    x_val_orig, _, drift_ids = make_feature_matrix(val_samples, MODALITY, original_norm, FEATURE_SET)

    drift_rows = []
    for recon_name, recon_root in RECON_ROOTS.items():
        recon_norm = recon_feature_norms.get(recon_name, 'none')
        recon_val_samples = samples_by_ids(recon_root, drift_ids, MODALITY)
        x_val_recon, _, _ = make_feature_matrix(recon_val_samples, MODALITY, recon_norm, FEATURE_SET)
        diff = x_val_recon - x_val_orig
        for idx, sample_id in enumerate(drift_ids):
            drift_rows.append({
                'source': recon_name,
                'sample_id': str(sample_id),
                'original_feature_normalization': original_norm,
                'recon_feature_normalization': recon_norm,
                'feature_mae': float(np.mean(np.abs(diff[idx]))),
                'feature_rmse': float(np.sqrt(np.mean(diff[idx] ** 2))),
                'feature_max_abs': float(np.max(np.abs(diff[idx]))),
                'feature_l2': float(np.linalg.norm(diff[idx])),
            })
    feature_drift_df = pd.DataFrame(drift_rows)
    out = DIAG_DIR / 'feature_drift_by_sample.csv'
    feature_drift_df.to_csv(out, index=False)
    print('Saved:', out)
    display(feature_drift_df.groupby('source')[['feature_mae', 'feature_rmse', 'feature_max_abs', 'feature_l2']].describe())

    if not predictions_df.empty and not feature_drift_df.empty:
        pred_sample_errors = (
            predictions_df[predictions_df['mode'].eq('original_train_to_recon_val')]
            .groupby(['source', 'model', 'sample_id'], dropna=False)
            .agg(prediction_mae=('abs_error', 'mean'), prediction_mse=('squared_error', 'mean'))
            .reset_index()
        )
        drift_join = pred_sample_errors.merge(feature_drift_df, on=['source', 'sample_id'], how='inner')
        corr_rows = []
        for keys, group in drift_join.groupby(['source', 'model'], dropna=False):
            row = {'source': keys[0], 'model': keys[1], 'samples': len(group)}
            for drift_col in ['feature_mae', 'feature_rmse', 'feature_max_abs', 'feature_l2']:
                for pred_col in ['prediction_mae', 'prediction_mse']:
                    if len(group) < 2 or group[drift_col].std() == 0 or group[pred_col].std() == 0:
                        corr = np.nan
                    else:
                        corr = float(group[drift_col].corr(group[pred_col]))
                    row[f'corr_{drift_col}_vs_{pred_col}'] = corr
            corr_rows.append(row)
        drift_corr_df = pd.DataFrame(corr_rows)
        out = DIAG_DIR / 'feature_drift_prediction_error_correlations.csv'
        drift_corr_df.to_csv(out, index=False)
        print('Saved:', out)
        display(drift_corr_df.sort_values(['source', 'model']).reset_index(drop=True))

        for _, best_row in best_df[(~best_df['source'].eq('original')) & best_df['mode'].eq('original_train_to_recon_val')].iterrows():
            source = best_row['source']
            model = best_row['model']
            plot_df = drift_join[drift_join['source'].eq(source) & drift_join['model'].eq(model)]
            if plot_df.empty:
                continue
            fig, ax = plt.subplots(figsize=(5.5, 4))
            ax.scatter(plot_df['feature_mae'], plot_df['prediction_mae'], s=14, alpha=0.65)
            ax.set_xlabel('feature MAE: reconstruction vs original')
            ax.set_ylabel('prediction MAE')
            ax.set_title(f'Feature drift vs prediction error: {source} | {model}')
            ax.grid(True, alpha=0.25)
            out = DIAG_DIR / f'{source}_{model}_feature_drift_vs_prediction_error.png'
            fig.savefig(out, bbox_inches='tight')
            print('Saved:', out)
            plt.show()


## 10. Sweep normalizacji cech: `none` vs `percentile`


In [ ]:
sweep_rows = []
if RUN_NORMALIZATION_SWEEP:
    original_samples = build_hyperview2_samples(HV2_ROOT, modality=MODALITY, split='train')
    train_samples, val_samples = split_samples(original_samples, val_fraction=VAL_FRACTION, seed=SEED)
    train_ids = [sample.sample_id for sample in train_samples]
    val_ids_for_sweep = [sample.sample_id for sample in val_samples]

    for normalization in NORMALIZATION_GRID:
        x_train_orig, y_train, _ = make_feature_matrix(train_samples, MODALITY, normalization, FEATURE_SET)
        x_val_orig, y_val, _ = make_feature_matrix(val_samples, MODALITY, normalization, FEATURE_SET)
        baseline_mse = ((y_val - y_train.mean(axis=0, keepdims=True)) ** 2).mean(axis=0).astype(np.float32)
        sweep_rows.extend(run_regressors(
            x_train_orig,
            y_train,
            x_val_orig,
            y_val,
            baseline_mse,
            MODEL_NAMES,
            'original',
            'original_train_to_original_val',
            normalization,
        ))

        for recon_name, recon_root in RECON_ROOTS.items():
            recon_train = samples_by_ids(recon_root, train_ids, MODALITY)
            recon_val = samples_by_ids(recon_root, val_ids_for_sweep, MODALITY)
            recon_feature_normalization = (
                'none'
                if RECON_INPUT_NORMALIZATIONS.get(recon_name) == normalization
                else normalization
            )
            x_train_recon, y_train_recon, _ = make_feature_matrix(
                recon_train,
                MODALITY,
                recon_feature_normalization,
                FEATURE_SET,
            )
            x_val_recon, y_val_recon, _ = make_feature_matrix(
                recon_val,
                MODALITY,
                recon_feature_normalization,
                FEATURE_SET,
            )
            sweep_rows.extend(run_regressors(
                x_train_orig,
                y_train,
                x_val_recon,
                y_val_recon,
                baseline_mse,
                MODEL_NAMES,
                recon_name,
                'original_train_to_recon_val',
                normalization,
                recon_feature_normalization=recon_feature_normalization,
            ))
            sweep_rows.extend(run_regressors(
                x_train_recon,
                y_train_recon,
                x_val_recon,
                y_val_recon,
                baseline_mse,
                MODEL_NAMES,
                recon_name,
                'recon_train_to_recon_val',
                normalization,
                recon_feature_normalization=recon_feature_normalization,
            ))

sweep_df = pd.DataFrame(sweep_rows)
if not sweep_df.empty:
    out = DIAG_DIR / 'normalization_sweep.csv'
    sweep_df.to_csv(out, index=False)
    print('Saved:', out)
    display(sweep_df.sort_values(['normalization', 'source', 'mode', 'hyperview_score']).reset_index(drop=True))
else:
    print('RUN_NORMALIZATION_SWEEP=False or no rows produced.')


In [ ]:
if 'sweep_df' in globals() and not sweep_df.empty:
    # best_by_mode ignores normalization, so compute explicitly here.
    ok = sweep_df[sweep_df['status'].eq('ok') & sweep_df['hyperview_score'].notna()].copy()
    idx = ok.groupby(['normalization', 'source', 'mode'])['hyperview_score'].idxmin()
    best_sweep = ok.loc[idx].sort_values(['source', 'mode', 'normalization']).reset_index(drop=True)
    display(best_sweep[['normalization', 'source', 'mode', 'model', 'hyperview_score', 'mean_mse', 'mean_mae']])

    fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(best_sweep))))
    labels = best_sweep['source'] + ' | ' + best_sweep['mode'] + ' | ' + best_sweep['normalization']
    y = np.arange(len(best_sweep))
    ax.barh(y, best_sweep['hyperview_score'], color='#59a14f')
    ax.axvline(1.0, color='black', linestyle='--', linewidth=1)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_xlabel('Best Hyperview score')
    ax.set_title('Normalization sweep summary')
    ax.grid(True, axis='x', alpha=0.25)
    fig.tight_layout()
    out = DIAG_DIR / 'normalization_sweep_best_scores.png'
    fig.savefig(out, bbox_inches='tight')
    print('Saved:', out)
    plt.show()



## 11. Eksport artefaktow


In [ ]:
print('Diagnostics directory:', DIAG_DIR)
for path in sorted(DIAG_DIR.glob('*')):
    if path.is_file():
        print(path.name, f'{path.stat().st_size / 1024:.1f} KiB')

